# Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Standard library
import os
import csv
import json
from pathlib import Path
from joblib import dump, load
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union, NamedTuple

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.signal import butter, detrend, filtfilt, iirnotch, medfilt, resample, welch
from scipy.stats import skew
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve
)
from sklearn.model_selection import train_test_split

# Keras / TensorFlow
from tensorflow.keras.callbacks import (
    TensorBoard, ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, History
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    Dense, Dropout, BatchNormalization, Flatten
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Loading Functions

In [ ]:
def load_ppg_data(
    root_dir: str,
    *,
    ext: str = "txt",
    sep: str = "_",
    recursive: bool = True
) -> Tuple[List[np.ndarray], List[str]]:
    """
    Load PPG signals from .txt files in per-subject subfolders.

    Args:
        root_dir: Path to the root directory containing one folder per subject.
        ext:      File extension to look for (default "txt").
        sep:      Separator used in filenames to split off the label (default "_").
                  The text before the first `sep` is taken as the label.
        recursive: If True, descends into nested subdirectories; otherwise only one level.

    Returns:
        signals: List of 1D numpy arrays, one per file.
        labels:  List of string labels corresponding to each signal.
    """
    root = Path(root_dir)
    if not root.is_dir():
        raise ValueError(f"{root_dir!r} is not a valid directory")

    signals: List[np.ndarray] = []
    labels:  List[str]       = []

    pattern = f"**/*.{ext}" if recursive else f"*.{ext}"
    for subject_dir in root.iterdir():
        if not subject_dir.is_dir():
            continue

        for file in subject_dir.glob(pattern):
            # derive label from filename
            label = file.stem.split(sep, 1)[0]

            # load floats; skip invalid lines
            try:
                data = np.genfromtxt(
                    file, dtype=float, comments=None, invalid_raise=False
                )
            except Exception:
                continue

            # genfromtxt returns a scalar for single-line files
            arr = np.atleast_1d(data).astype(float)
            # drop NaNs from parsing errors
            arr = arr[~np.isnan(arr)]

            if arr.size == 0:
                continue

            signals.append(arr)
            labels.append(label)

    return signals, labels

# Signal Processing Functions

In [ ]:
def butter_bandpass(lowcut: float, highcut: float, fs: float, order: int = 4) -> Tuple[np.ndarray, np.ndarray]:
    """Design a Butterworth bandpass filter."""
    nyq = fs / 2
    low = lowcut / nyq
    high = highcut / nyq
    return butter(order, [low, high], btype='band')

In [ ]:
def apply_notch(signal: np.ndarray, fs: float, freq: float = 50.0, q: float = 30.0) -> np.ndarray:
    """Apply an IIR notch filter at given frequency."""
    nyq = fs / 2
    b, a = iirnotch(freq / nyq, q)
    return filtfilt(b, a, signal)

In [ ]:
def highpass_exponential(signal: np.ndarray, fs: float, tc: float) -> np.ndarray:
    """
    Apply a first-order highpass filter via exponential smoothing.
    tc: time constant in seconds.
    """
    alpha = 1 / (1 + 1/(fs * tc))
    y = np.empty_like(signal)
    y[0] = signal[0]
    for i in range(1, len(signal)):
        y[i] = alpha * (signal[i] + y[i-1] - (1-alpha) * y[i-1])

    return y

In [ ]:
def moving_average(signal: np.ndarray, window: int) -> np.ndarray:
    """Compute a moving average with a flat window."""
    return np.convolve(signal, np.ones(window)/window, mode='same')

In [ ]:
def normalize_minmax(signal: np.ndarray) -> np.ndarray:
    """Rescale signal to [0,1]."""
    mn, mx = signal.min(), signal.max()
    return (signal - mn) / (mx - mn) if mx > mn else signal - mn

In [ ]:
def preprocess_ppg(
    signal: np.ndarray,
    fs: float = 100.0,
    bandpass: Tuple[float, float] = (0.5, 8.0),
    notch: Tuple[float, float] = (50.0, 30.0),
    median_kernel: int = 5,
    hp_tc: Optional[float] = 0.2,
    smooth_kernel: Optional[np.ndarray] = None,
    detrend_window_s: float = 5.0,
    do_minmax: bool = True
) -> np.ndarray:
    """
    Preprocess a raw PPG signal:
      1. Bandpass Butterworth
      2. Notch filter
      3. Median filter
      4. Optional first-order highpass
      5. Smoothing via convolution
      6. Detrending by subtracting moving average
      7. Optional min-max normalization

    Args:
        signal:          1D raw PPG signal.
        fs:              Sampling rate (Hz).
        bandpass:        (lowcut, highcut) frequencies for bandpass.
        notch:           (freq, Q) for notch filter.
        median_kernel:   Kernel size for median filter.
        hp_tc:           Time constant (s) for highpass; if None, skip HP.
        smooth_kernel:   1D array of smoothing weights; if None, uses default.
        detrend_window_s: Window length (s) for moving-average detrending.
        do_minmax:       Whether to min-max normalize final output.

    Returns:
        Preprocessed 1D signal, length reduced by detrend window.
    """
    # 1) Bandpass
    b_bp, a_bp = butter_bandpass(*bandpass, fs=fs)
    x = filtfilt(b_bp, a_bp, signal)

    # 2) Notch
    x = apply_notch(x, fs, *notch)

    # 3) Median filter
    x = medfilt(x, kernel_size=median_kernel)

    # 4) Highpass
    if hp_tc is not None:
        x = highpass_exponential(x, fs, hp_tc)

    # 5) Smoothing
    if smooth_kernel is None:
        smooth_kernel = np.array([0.025,0.04,0.07,0.13,0.2,0.22,0.2,0.13,0.07,0.04,0.025])
        smooth_kernel = smooth_kernel / smooth_kernel.sum()
    x = np.convolve(x, smooth_kernel, mode='same')

    # 6) Detrend
    window = int(fs * detrend_window_s)
    trend = moving_average(x, window)
    x = x[window:] - trend[window:]

    # 7) Normalize
    if do_minmax:
        x = normalize_minmax(x)

    return x

In [ ]:
from scipy.signal import butter, filtfilt, iirnotch, resample_poly

def resample_signal(signal, original_fs, target_fs, anti_aliasing=True, filter_order=5):
    """
    Resample a signal to a target sampling frequency.

    Parameters:
    - signal: 1D numpy array, the input signal.
    - original_fs: float, original sampling frequency in Hz.
    - target_fs: float, desired target sampling frequency in Hz.
    - anti_aliasing: bool, whether to apply anti-aliasing filter when downsampling.
    - filter_order: int, order of the anti-aliasing filter.

    Returns:
    - resampled_signal: 1D numpy array, the resampled signal.
    """
    if original_fs == target_fs:
        print("Original and target sampling frequencies are the same. No resampling applied.")
        return signal.copy()

    # Calculate the greatest common divisor of the two sampling rates
    from math import gcd
    gcd_fs = gcd(int(original_fs), int(target_fs))
    up = target_fs // gcd_fs
    down = original_fs // gcd_fs

    if anti_aliasing and down > 1:
        # Design a low-pass Butterworth filter to prevent aliasing
        nyquist_target = 0.5 * target_fs
        cutoff = nyquist_target * 0.9  # 90% of the Nyquist frequency of the target
        normal_cutoff = cutoff / (0.5 * original_fs)
        b, a = butter(filter_order, normal_cutoff, btype='low', analog=False)
        filtered_signal = filtfilt(b, a, signal)
    else:
        filtered_signal = signal.copy()

    # Resample using polyphase filtering
    resampled_signal = resample_poly(filtered_signal, up, down)

    return resampled_signal

In [ ]:
def resample_signals(signals, original_fs, target_fs, anti_aliasing=True, filter_order=5):
    """
    Resample multiple signals to a target sampling frequency.

    Parameters:
    - signals: 2D numpy array or list of 1D arrays, where each sub-array is a signal.
    - original_fs: float, original sampling frequency in Hz.
    - target_fs: float, desired target sampling frequency in Hz.
    - anti_aliasing: bool, whether to apply anti-aliasing filter when downsampling.
    - filter_order: int, order of the anti-aliasing filter.

    Returns:
    - resampled_signals: 2D numpy array, resampled signals.
    """
    resampled_signals = []

    for idx, signal in enumerate(signals):
        print(f"Resampling signal {idx+1}/{len(signals)}")
        resampled = resample_signal(signal, original_fs, target_fs, anti_aliasing, filter_order)
        resampled_signals.append(resampled)

    return resampled_signals

# Splitting Functions

In [ ]:
def sliding_windows(
    signal: np.ndarray,
    window_size: int,
    overlap: float
) -> List[np.ndarray]:
    """
    Generate overlapping windows from a 1D signal.

    Args:
        signal:       1D numpy array.
        window_size:  Number of samples per window.
        overlap:      Fractional overlap between [0,1).

    Returns:
        List of 1D numpy arrays, each of length `window_size`.
    """
    if not (0 <= overlap < 1):
        raise ValueError("overlap must be in [0,1)")
    step = int(window_size * (1 - overlap))
    if step <= 0:
        raise ValueError("window_size and overlap produce non-positive step")
    return [
        signal[i : i + window_size]
        for i in range(0, len(signal) - window_size + 1, step)
    ]

In [ ]:
def split_and_window(
    signals: List[np.ndarray],
    labels: List[Any],
    window_size: int,
    overlap: float,
    train_frac: float,
    val_frac: float,
    seed: int = 42
) -> Tuple[
    np.ndarray, np.ndarray,
    np.ndarray, np.ndarray,
    np.ndarray, np.ndarray
]:
    """
    For each subject (signal, label), split the raw signal into
    train/val/test segments and then sliding-window each segment.

    Returns:
      X_tr, y_tr, X_val, y_val, X_te, y_te
      where X_* have shape (n_windows, window_size, 1)
            y_* have shape (n_windows,)
    """
    np.random.seed(seed)

    X_tr_list, y_tr_list = [], []
    X_val_list, y_val_list = [], []
    X_te_list, y_te_list   = [], []

    for sig, lbl in zip(signals, labels):
        n = len(sig)
        i_tr  = int(n * train_frac)
        i_val = int(n * (train_frac + val_frac))

        # raw segments
        seg_tr, seg_val, seg_te = sig[:i_tr], sig[i_tr:i_val], sig[i_val:]

        # window and label them
        for win in sliding_windows(seg_tr, window_size, overlap):
            X_tr_list.append(win);  y_tr_list.append(lbl)
        for win in sliding_windows(seg_val, window_size, overlap):
            X_val_list.append(win); y_val_list.append(lbl)
        for win in sliding_windows(seg_te, window_size, overlap):
            X_te_list.append(win);  y_te_list.append(lbl)

    # stack and add channel dim
    X_tr = np.array(X_tr_list)[..., None]
    X_val= np.array(X_val_list)[..., None]
    X_te = np.array(X_te_list)[..., None]
    y_tr, y_val, y_te = map(np.array, (y_tr_list, y_val_list, y_te_list))

    return X_tr, y_tr, X_val, y_val, X_te, y_te

# Filtering Functions

In [ ]:
class QCCuts(NamedTuple):
    var_min: float
    snr_min: float
    skew_min: float
    skew_max: float

In [ ]:
def window_snr(
    signal: np.ndarray,
    fs: float = 100.0,
    f_low: float = 0.5,
    f_high: float = 9.0
) -> float:
    """
    Estimate the signal-to-noise ratio (SNR) of a window using Welch's method.

    The "signal" power is the PSD integrated between f_low and f_high,
    and the "noise" power is the remainder of the PSD.

    Parameters
    ----------
    signal : np.ndarray
        1D array of time-domain samples.
    fs : float, default=100.0
        Sampling frequency in Hz.
    f_low : float, default=0.5
        Lower bound of the signal band (Hz).
    f_high : float, default=9.0
        Upper bound of the signal band (Hz).

    Returns
    -------
    float
        Estimated SNR (linear scale). Higher is better signal quality.
    """
    # Compute power spectral density
    nperseg = int(fs)
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)

    # Mask for the signal band
    in_band = (freqs >= f_low) & (freqs <= f_high)

    # Integrate PSD to get band and total power
    power_signal = float(np.sum(psd[in_band]))
    power_total  = float(np.sum(psd))

    # Ensure noise floor isn't zero
    power_noise = max(power_total - power_signal, 1e-16)

    return power_signal / power_noise

In [ ]:
def compute_qc_metrics(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute QC metrics (variance, SNR, skewness) on each window.

    Args:
        X: array of shape (n_windows, window_length, 1).

    Returns:
        v: array of variances, shape (n_windows,).
        s: array of SNRs,       shape (n_windows,).
        k: array of skewness,   shape (n_windows,).
    """
    # remove channel dimension
    windows = X.squeeze(-1)
    v = windows.var(axis=1)
    s = np.array([window_snr(win) for win in windows])
    k = np.array([skew(win)       for win in windows])
    return v, s, k

In [ ]:
def compute_cuts(
    X: np.ndarray,
    pctiles: Tuple[float, float, float, float] = (5, 5, 5, 95)
) -> QCCuts:
    """
    Determine QC cutoff values for variance, SNR, and skewness.

    Args:
        X:        array (n_windows, window_length, 1) to derive cuts from.
        pctiles:  (var_pct, snr_pct, skew_low_pct, skew_high_pct).

    Returns:
        QCCuts namedtuple with fields (var_min, snr_min, skew_min, skew_max).
    """
    v, s, k = compute_qc_metrics(X)
    var_min      = np.percentile(v, pctiles[0])
    snr_min      = np.percentile(s, pctiles[1])
    skew_min     = np.percentile(k, pctiles[2])
    skew_max     = np.percentile(k, pctiles[3])
    return QCCuts(var_min, snr_min, skew_min, skew_max)

In [ ]:
def qc_filter(
    X: np.ndarray,
    cuts: QCCuts
) -> np.ndarray:
    """
    Generate a boolean mask of windows that pass QC thresholds.

    Args:
        X:    array (n_windows, window_length, 1).
        cuts: QCCuts(var_min, snr_min, skew_min, skew_max).

    Returns:
        mask: boolean array of shape (n_windows,), True if window passes all cuts.
    """
    v, s, k = compute_qc_metrics(X)
    return (
        (v > cuts.var_min) &
        (s > cuts.snr_min) &
        (k > cuts.skew_min) &
        (k < cuts.skew_max)
    )

# Augmentation Functions

In [ ]:
def add_noise(signal: np.ndarray, noise_level: float = 0.02) -> np.ndarray:
    """
    Add zero-mean Gaussian noise.
    :param signal: 1D input array.
    :param noise_level: standard deviation of Gaussian noise.
    """
    noise = np.random.normal(0, noise_level, size=signal.shape)
    return signal + noise

In [ ]:
def time_shift(signal: np.ndarray, shift_frac: float = 0.02) -> np.ndarray:
    """
    Circularly shift the signal by a random fraction of its length.
    :param signal: input 1D array.
    :param shift_frac: max fraction to shift (±).
    """
    n = len(signal)
    max_shift = int(n * shift_frac)
    if max_shift < 1:
        return signal.copy()
    shift = np.random.randint(-max_shift, max_shift)
    return np.roll(signal, shift)

In [ ]:
def jitter(signal: np.ndarray, jitter_level: float = 0.01) -> np.ndarray:
    """
    Add small uniform jitter to each sample.
    :param signal: input 1D array.
    :param jitter_level: max absolute perturbation.
    """
    return signal + np.random.uniform(-jitter_level, jitter_level, size=signal.shape)

In [ ]:
def time_warp(signal: np.ndarray, warp_frac: float = 0.2) -> np.ndarray:
    """
    Speed up or slow down the signal by a random factor.
    :param signal: input 1D array.
    :param warp_frac: maximum fractional change in length.
    """
    f = np.random.uniform(1 - warp_frac, 1 + warp_frac)
    # resample to new length then back to original
    warped = resample(signal, int(len(signal) / f))
    return resample(warped, len(signal))

In [ ]:
def freq_mask(signal: np.ndarray, mask_frac: float = 0.1) -> np.ndarray:
    """
    Zero out a random contiguous band in the FFT domain.
    :param signal: input 1D array.
    :param mask_frac: fraction of frequency bins to mask.
    """
    sp = np.fft.rfft(signal)
    n = sp.size
    w = int(n * mask_frac)
    if w < 1:
        return signal.copy()
    start = np.random.randint(0, n - w)
    sp[start:start + w] = 0
    return np.fft.irfft(sp, n=signal.size)

In [ ]:
AugmentBatchFn = Callable[[np.ndarray, np.ndarray, int], Tuple[np.ndarray, np.ndarray]]
AugmentSignalFn = Callable[[np.ndarray], np.ndarray]
def apply_augmentations(
    signal: np.ndarray,
    funcs: List[AugmentSignalFn]
) -> np.ndarray:
    """
    Sequentially apply each augmentation function in funcs to a 1D signal.
    """
    out = signal.copy()
    for fn in funcs:
        out = fn(out)
    return out

In [ ]:


def class_specific_augment(
    X: np.ndarray,
    y: np.ndarray,
    factors: Dict[Any, int],
    augment_funcs: List[AugmentSignalFn],
    default_factor: int = 1
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Oversample each class by generating `factor` augmented windows per original window.

    Args:
        X:             (n_samples, window_size, 1) input windows.
        y:             (n_samples,) integer labels.
        factors:       mapping cls -> augment factor, 'other' key for default.
        augment_funcs: list of functions that take a 1D np.ndarray and return augmented 1D.
        default_factor: fallback factor for classes not in factors.

    Returns:
        X_aug: (n_new, window_size, 1)
        y_aug: (n_new,)
    """
    window_size = X.shape[1]
    X_out, y_out = [], []

    for cls in np.unique(y):
        idx        = np.where(y == cls)[0]
        originals  = X[idx].reshape(-1, window_size)  # (m, w)
        factor     = factors.get(cls, factors.get('other', default_factor))

        # retain originals
        combined = [originals]
        labels   = [np.full(len(originals), cls)]

        # for each original, generate `factor` augmentations
        aug_list = []
        for win in originals:
            for _ in range(factor):
                aug_win = apply_augmentations(win, augment_funcs)
                aug_list.append(aug_win)
        if aug_list:
            aug_array = np.stack(aug_list, axis=0)
            combined.append(aug_array)
            labels.append(np.full(len(aug_list), cls))

        # collect per-class
        class_X = np.vstack(combined).reshape(-1, window_size, 1)
        class_y = np.concatenate(labels)
        X_out.append(class_X)
        y_out.append(class_y)

    # concatenate all classes
    X_aug = np.vstack(X_out)
    y_aug = np.concatenate(y_out)
    return X_aug, y_aug

# Thresholding Functions

In [ ]:
def find_best_thresholds(model, X, y_true_cat):
    """
    For multi‐class (emotion) threshold tuning.  If model.predict returns a list,
    we grab the first element (the emotion head) before computing thresholds.
    """
    raw = model.predict(X)
    probs = raw[0] if isinstance(raw, list) else raw  # emotion head only

    true_labels = np.argmax(y_true_cat, axis=1)
    thresholds  = {}
    for c in range(probs.shape[1]):
        p, r, t = precision_recall_curve((true_labels==c).astype(int), probs[:,c])
        f1      = 2*p*r/(p+r+1e-8)
        best    = np.nanargmax(f1[:-1])
        thresholds[c] = float(t[best])
    return thresholds

In [ ]:
def apply_thresholds(model, X, thresholds):
    raw   = model.predict(X)
    probs = raw[0] if isinstance(raw, list) else raw

    preds = np.full(len(probs), -1, dtype=int)
    for c, thr in thresholds.items():
        preds[probs[:,c] >= thr] = c

    # fallback to argmax
    miss = preds < 0
    if np.any(miss):
        preds[miss] = np.argmax(probs[miss], axis=1)
    return preds

# Evaluation and Saving functions

In [ ]:
def plot_confusion_matrix(
    cm: np.ndarray,
    labels: Sequence[Union[str, int]],
    save_path: Union[str, Path],
    figsize: Tuple[int, int] = (6, 6),
    cmap: str = 'Blues'
) -> None:
    """
    Save a confusion matrix heatmap to disk.

    Args:
        cm:        Confusion matrix array of shape (n_labels, n_labels).
        labels:    Sequence of label names/integers.
        save_path: File path to save the PNG.
        figsize:   Figure size in inches.
        cmap:      Matplotlib colormap name.
    """
    save_path = Path(save_path)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, cmap=cmap)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    thresh = cm.max() / 2.0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm[i, j] > thresh else 'black'
            ax.text(j, i, f"{cm[i, j]}", ha='center', va='center', color=color)

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

In [ ]:
def evaluate_and_save(
    model,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    output_dir: Union[str, Path],
    model_name: str,
    task: str,
    num_classes: Optional[int] = None
) -> None:
    base = Path(output_dir)
    base.mkdir(parents=True, exist_ok=True)

    csv_path = base / f"{model_name}_metrics.csv"
    with csv_path.open('w', newline='') as cf:
        writer = csv.writer(cf)
        writer.writerow(['head','split','accuracy','precision','recall','f1','auc'])

        def _save_split(name, split, y_true, y_score, y_pred, n_cls, auc):
            # text report
            rpt = classification_report(y_true, y_pred, zero_division=0)
            (base/f"{model_name}_{name}_{split}_report.txt")\
              .write_text(rpt + f"\nAUC: {auc:.4f}\n")

            # confusion matrix
            cm = confusion_matrix(y_true, y_pred)
            plot_confusion_matrix(cm, list(range(cm.shape[0])),
                                  base/f"{model_name}_{name}_{split}_cm.png")

            # summary metrics
            acc  = accuracy_score(y_true, y_pred)
            prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
            rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
            f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
            writer.writerow([name, split,
                             f"{acc:.4f}", f"{prec:.4f}",
                             f"{rec:.4f}", f"{f1:.4f}", f"{auc:.4f}"])

        # EMOTION head (multi‐class)
        if task in ('emotion','multi'):
            assert num_classes is not None
            # tune on validation
            y_val_cat = to_categorical(y_val, num_classes)
            thr       = find_best_thresholds(model, X_val, y_val_cat)
            thr_py    = {int(c): float(t) for c,t in thr.items()}
            (base/"thresholds.json").write_text(json.dumps(thr_py, indent=2))

            # get probs & preds
            raw_val    = model.predict(X_val)
            probs_val  = raw_val[0] if isinstance(raw_val, list) else raw_val
            raw_test   = model.predict(X_test)
            probs_test = raw_test[0] if isinstance(raw_test, list) else raw_test

            y_val_pred  = apply_thresholds(model, X_val, thr)
            y_test_pred = apply_thresholds(model, X_test, thr)

            auc_val     = roc_auc_score(y_val_cat, probs_val, multi_class='ovr')
            auc_test    = roc_auc_score(to_categorical(y_test,num_classes), probs_test, multi_class='ovr')

            _save_split('emotion','val',  y_val,   probs_val,  y_val_pred,  num_classes, auc_val)
            _save_split('emotion','test', y_test,  probs_test, y_test_pred, num_classes, auc_test)

        # VALENCE head
        if task in ('valence','multi'):
            map_val    = {0:0,1:1,2:1,3:0}
            y_val_v    = np.vectorize(map_val.get)(y_val)
            y_test_v   = np.vectorize(map_val.get)(y_test)
            probs_val  = model.predict(X_val)[1] if task=='multi' else model.predict(X_val)
            probs_test = model.predict(X_test)[1] if task=='multi' else model.predict(X_test)

            y_val_pred  = (probs_val.flatten() >= 0.5).astype(int)
            y_test_pred = (probs_test.flatten()>= 0.5).astype(int)
            auc_val     = roc_auc_score(y_val_v, probs_val.flatten())
            auc_test    = roc_auc_score(y_test_v, probs_test.flatten())

            _save_split('valence','val',  y_val_v,   probs_val,  y_val_pred,  None, auc_val)
            _save_split('valence','test', y_test_v,  probs_test, y_test_pred, None, auc_test)

        # AROUSAL head
        if task in ('arousal','multi'):
            map_aro    = {0:1,1:1,2:0,3:0}
            y_val_a    = np.vectorize(map_aro.get)(y_val)
            y_test_a   = np.vectorize(map_aro.get)(y_test)
            probs_val  = model.predict(X_val)[2] if task=='multi' else model.predict(X_val)
            probs_test = model.predict(X_test)[2] if task=='multi' else model.predict(X_test)

            y_val_pred  = (probs_val.flatten() >= 0.5).astype(int)
            y_test_pred = (probs_test.flatten()>= 0.5).astype(int)
            auc_val     = roc_auc_score(y_val_a, probs_val.flatten())
            auc_test    = roc_auc_score(y_test_a, probs_test.flatten())

            _save_split('arousal','val',  y_val_a,   probs_val,  y_val_pred,  None, auc_val)
            _save_split('arousal','test', y_test_a,  probs_test, y_test_pred, None, auc_test)

In [ ]:
def save_history(history: History, output_dir: str, model_name: str) -> None:
    """
    Save a Keras History object to disk as JSON and plot each metric’s curve.

    Args:
        history:    The History object returned by model.fit().
        output_dir: Directory where files will be saved.
        model_name: Prefix for the history JSON and metric plots.
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1) Save raw history to JSON
    hist_path = os.path.join(output_dir, f"{model_name}_history.json")
    with open(hist_path, 'w') as f:
        json.dump(history.history, f)

    # 2) Plot each metric over epochs
    for metric, values in history.history.items():
        plt.figure()
        plt.plot(values)
        plt.title(f"{metric} over epochs")
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{model_name}_{metric}.png"), bbox_inches='tight')
        plt.close()

# Training functions

In [ ]:
def save_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        json.dump(data, f, indent=2)

In [ ]:
def build_callbacks(
    output_dir: Path,
    model_name: str,
    monitor: str = "val_loss",
    patience: int = 10,
    lr_patience: int = 5,
    lr_factor: float = 0.5,
    min_lr: float = 1e-6,
    log_histogram_freq: int = 1
) -> list[tf.keras.callbacks.Callback]:
    """
    Create a standard set of training callbacks:
      - TensorBoard logging
      - ModelCheckpoint (best model by `monitor`)
      - EarlyStopping
      - ReduceLROnPlateau
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = output_dir / f"{model_name}_best.keras"
    log_dir   = output_dir / "logs"

    return [
        TensorBoard(log_dir=str(log_dir), histogram_freq=log_histogram_freq),
        ModelCheckpoint(
            filepath=str(ckpt_path),
            monitor=monitor,
            mode="min",
            save_best_only=True,
            verbose=1
        ),
        EarlyStopping(
            monitor=monitor,
            mode="min",
            patience=patience,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor=monitor,
            mode="min",
            factor=lr_factor,
            patience=lr_patience,
            min_lr=min_lr,
            verbose=1
        )
    ]

In [ ]:
def train_and_save(
    model: Model,
    X_train: tf.Tensor,
    y_train: tf.Tensor,
    X_val: tf.Tensor,
    y_val: tf.Tensor,
    *,
    output_dir: Path,
    model_name: str,
    batch_size: int = 32,
    epochs: int = 50,
    monitor: str = "val_loss"
) -> tf.keras.callbacks.History:
    """
    Compile (if needed), fit `model` on the training set with validation,
    and save the best weights via callbacks.

    Args:
        model:       A compiled Keras Model.
        X_train:     Training inputs.
        y_train:     Training labels.
        X_val:       Validation inputs.
        y_val:       Validation labels.
        output_dir:  Directory where callbacks will save logs and checkpoints.
        model_name:  Prefix for checkpoint files.
        batch_size:  Batch size for training.
        epochs:      Max number of epochs.
        monitor:     Quantity to monitor for early stopping and checkpoint.

    Returns:
        The History object returned by `model.fit`.
    """
    # Ensure output exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Prepare callbacks
    cbs = build_callbacks(output_dir, model_name, monitor=monitor)

    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=cbs,
        verbose=1
    )

    return history

In [ ]:
Signals  = List[np.ndarray]
Labels   = np.ndarray
ModelFn  = Callable[[Tuple[int,int], int, Labels], Any]

In [ ]:
def prepare_window_data(
    signals: Signals,
    labels: Labels,
    window_size: int,
    overlap: float,
    train_frac: float,
    val_frac: float,
    qc_pctiles: Tuple[float, float, float, float],
    augment_factors: Dict[Any, int],
    augment_funcs: List[AugmentSignalFn],
    seed: int
) -> Tuple[
    np.ndarray, Labels,
    np.ndarray, Labels,
    np.ndarray, Labels,
    int
]:
    """
    1) Subject‐dependent split & sliding‐window
    2) QC filter (variance/SNR/skew)
    3) Class‐specific augmentation
    4) Standardize (fit on train, apply to all splits)

    Returns:
        X_tr, y_tr, X_val, y_val, X_te, y_te, n_classes
    """
    # A) split & window
    X_tr, y_tr, X_val, y_val, X_te, y_te = split_and_window(
        signals, labels,
        window_size=window_size,
        overlap=overlap,
        train_frac=train_frac,
        val_frac=val_frac,
        seed=seed
    )

    # B) QC filter
    cuts    = compute_cuts(X_tr, qc_pctiles)
    mask_tr = qc_filter(X_tr, cuts)
    mask_va = qc_filter(X_val, cuts)
    mask_te = qc_filter(X_te, cuts)

    X_tr, y_tr   = X_tr[mask_tr], y_tr[mask_tr]
    X_val, y_val = X_val[mask_va], y_val[mask_va]
    X_te, y_te   = X_te[mask_te],   y_te[mask_te]

    # C) class‐specific augmentation (train only)
    if len(augment_factors) > 0:
      X_tr, y_tr = class_specific_augment(
          X_tr, y_tr,
          factors=augment_factors,
          augment_funcs=augment_funcs,
          default_factor=1
      )

    # D) standardize with sklearn.StandardScaler
    scaler = StandardScaler()
    # reshape to (n_samples, window_size)
    tr_flat = X_tr.reshape(-1, window_size)
    X_tr_scaled = scaler.fit_transform(tr_flat)
    X_val_flat  = X_val.reshape(-1, window_size)
    X_val_scaled = scaler.transform(X_val_flat)
    X_te_flat   = X_te.reshape(-1, window_size)
    X_te_scaled  = scaler.transform(X_te_flat)

    # reshape back to 3D (n_samples, window_size, 1)
    X_tr = X_tr_scaled.reshape(-1, window_size, 1)
    X_val= X_val_scaled.reshape(-1, window_size, 1)
    X_te = X_te_scaled.reshape(-1, window_size, 1)

    dump(scaler, 'scaler.joblib')

    # E) number of distinct classes
    n_cls = int(np.unique(labels).size)

    return X_tr, y_tr, X_val, y_val, X_te, y_te, n_cls

In [ ]:
def run_single_task(
    task: str,
    X_tr: np.ndarray, y_tr: Labels,
    X_val: np.ndarray, y_val: Labels,
    X_te: np.ndarray, y_te: Labels,
    n_cls: int,
    build_model_fn: ModelFn,
    output_root: Path,
    window_size: int,
    batch_size: int,
    epochs: int
) -> None:
    """
    Train & evaluate one task on preprocessed data.
    """
    model_name = f"{task}_{window_size}"
    out_dir    = output_root / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # ——— format labels per task ———
    if task == "emotion":
        # 4‐way one‐hot
        y_tr_in  = to_categorical(y_tr, n_cls)
        y_val_in = to_categorical(y_val, n_cls)

    elif task == "valence":
        # negative (anger, sadness)=0, positive (joy, relaxed)=1
        map_val = {0: 0, 1: 1, 2: 1, 3: 0}
        y_tr_in  = np.array([map_val[int(c)] for c in y_tr])
        y_val_in = np.array([map_val[int(c)] for c in y_val])

    elif task == "arousal":
        # high (anger, joy)=1, low (relaxed, sadness)=0
        map_ar  = {0: 1, 1: 1, 2: 0, 3: 0}
        y_tr_in  = np.array([map_ar[int(c)] for c in y_tr])
        y_val_in = np.array([map_ar[int(c)] for c in y_val])

    elif task == "multi":
        # multi‐output: dict of arrays
        y_tr_in = {
          "emotion":  to_categorical(y_tr, n_cls),
          "valence":  np.array([0 if c in (0,3) else 1 for c in y_tr]),
          "arousal":  np.array([1 if c in (0,1) else 0 for c in y_tr])
        }
        y_val_in = {
          "emotion":  to_categorical(y_val, n_cls),
          "valence":  np.array([0 if c in (0,3) else 1 for c in y_val]),
          "arousal":  np.array([1 if c in (0,1) else 0 for c in y_val])
        }

    else:
        raise ValueError(f"Unknown task {task}")

    # build, train, save
    model   = build_model_fn(X_tr.shape[1:], n_cls, y_tr)
    history = train_and_save(
        model,
        X_tr, y_tr_in,
        X_val, y_val_in,
        model_name=model_name,
        output_dir=out_dir,
        batch_size=batch_size,
        epochs=epochs
    )

    # save training artifacts
    save_json(out_dir / "history.json", history.history)
    save_history(history, str(out_dir), model_name)
    model.save(out_dir / f"{model_name}.keras")

    # evaluate
    evaluate_and_save(
        model,
        X_val, y_val,
        X_te, y_te,
        output_dir=str(out_dir),
        model_name=model_name,
        task=task,
        num_classes=(n_cls if task in ("emotion","multi") else None)
    )

In [ ]:
def run_experiments(
    root_dir: Path,
    second_dir: Optional[Path],
    window_sizes: List[int],
    tasks: List[str],
    qc_pctiles: Tuple[float,float,float,float],
    augment_factors: Dict[Any,int],
    augment_funcs: List[AugmentSignalFn],
    builders: Dict[str,ModelFn],
    output_root: Path = Path("./results"),
    overlap: float = 0.5,
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    batch_size: int = 32,
    epochs: int = 250,
    seed: int = 42
) -> None:
    """
    Load & preprocess once, then for each window size and augmentation setting:
      - prepare data (split → QC → optional augment)
      - for each task: train & evaluate
    """
    # 1) Load & preprocess raw
    raw_signals, raw_labels = load_ppg_data(root_dir)
    signals = [preprocess_ppg(sig) for sig in raw_signals]

    if second_dir is not None:
        raw_signals2, raw_labels2 = load_ppg_data(second_dir)
        signals2 = [resample_signal(preprocess_ppg(sig),176,100) for sig in raw_signals2]
        signals += signals2
        raw_labels += raw_labels2

    labels_enc = LabelEncoder().fit_transform(raw_labels)

    output_root.mkdir(parents=True, exist_ok=True)

    for win in window_sizes:
        # two modes: no augmentation, then augmentation
        for do_aug in (False,):
            suffix = "_aug" if do_aug else ""
            print(f"\n=== Window {win} {'with' if do_aug else 'without'} augmentation ===")

            # prepare data once for this mode
            X_tr, y_tr, X_val, y_val, X_te, y_te, n_cls = prepare_window_data(
                signals,
                labels_enc,
                window_size     = win,
                overlap         = overlap,
                train_frac      = train_frac,
                val_frac        = val_frac,
                qc_pctiles      = qc_pctiles,
                augment_factors = (augment_factors if do_aug else {}),
                augment_funcs   = (augment_funcs if do_aug else []),
                seed            = seed
            )

            # now loop tasks
            for task in tasks:
                # if win == 500 and task in ['emotion','valence','arousal'] and not do_aug:
                #     continue
                if task not in builders:
                    raise ValueError(f"Unknown task: {task!r}")
                print(f"--- Task={task} @ window={win}{suffix} ---")

                # choose subfolder for this mode
                out_dir = output_root / f"{task}_{win}{suffix}"
                out_dir.mkdir(parents=True, exist_ok=True)

                run_single_task(
                    task           = task,
                    X_tr           = X_tr,
                    y_tr           = y_tr,
                    X_val          = X_val,
                    y_val          = y_val,
                    X_te           = X_te,
                    y_te           = y_te,
                    n_cls          = n_cls,
                    build_model_fn = builders[task],
                    output_root    = out_dir,
                    window_size    = win,
                    batch_size     = batch_size,
                    epochs         = epochs
                )

# Models

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    GlobalAveragePooling1D, Dense, Dropout
)
import tensorflow as tf
import numpy as np
from tensorflow.keras.optimizers import Adam

# Shared backbone from the original multi-task CNN
def build_backbone(input_shape):
    inp = Input(shape=input_shape)
    x = Conv1D(64, 7, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.05))(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.1)(x)

    x = Conv1D(128, 6, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.05))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.1)(x)

    x = Conv1D(128, 5, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.05))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.)(x)

    x = GlobalAveragePooling1D()(x)
    x = Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.05))(x)
    x = Dropout(0.4)(x)

    return inp, x



# ———— 1) Focal loss for multi‐class ————
def focal_loss_multi(alpha, gamma=2.0):
    alpha = tf.constant(alpha, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        ce   = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
        p_t  = tf.reduce_sum(y_true * y_pred, axis=-1)
        a_t  = tf.reduce_sum(alpha * y_true, axis=-1)
        return a_t * tf.pow(1 - p_t, gamma) * ce
    return loss_fn

# ———— Emotion model ————
def create_emotion_model(input_shape, num_emotions, y_tr_int):
    freqs = np.bincount(y_tr_int, minlength=num_emotions)
    alpha = freqs.sum() / (num_emotions * freqs)
    inp, feat = build_backbone(input_shape)
    out = tf.keras.layers.Dense(num_emotions, activation='softmax', name='emotion')(feat)
    m = tf.keras.Model(inp, out)
    m.compile(
        optimizer=Adam(1e-4),
        loss=focal_loss_multi(alpha, gamma=2.0),
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return m

# ———— Valence model with binary focal ————
def create_valence_model(input_shape):
    inp, feat = build_backbone(input_shape)
    out = tf.keras.layers.Dense(1, activation='sigmoid', name='valence')(feat)
    m = tf.keras.Model(inp, out)
    m.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return m

# ———— Arousal model with binary focal ————
def create_arousal_model(input_shape):
    inp, feat = build_backbone(input_shape)
    out = tf.keras.layers.Dense(1, activation='sigmoid', name='arousal')(feat)
    m = tf.keras.Model(inp, out)
    m.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return m

# ———— Multi‐task with focal on all heads ————
def create_multi_task_cnn(input_shape, num_emotions, y_tr_int):
    # compute α for emotion head
    freqs_e = np.bincount(y_tr_int, minlength=num_emotions)
    alpha_e = freqs_e.sum() / (num_emotions * freqs_e)
    # compute α for valence/arousal heads
    # derive binary labels from emotion ints
    map_va = {0:(0,1), 1:(1,1), 2:(1,0), 3:(0,0)}
    y_tr_v = np.array([map_va[i][0] for i in y_tr_int])
    y_tr_a = np.array([map_va[i][1] for i in y_tr_int])
    freqs_v = np.bincount(y_tr_v, minlength=2)
    freqs_a = np.bincount(y_tr_a, minlength=2)
    alpha_v = 1.0 - (freqs_v[1] / freqs_v.sum())
    alpha_a = 1.0 - (freqs_a[1] / freqs_a.sum())

    # backbone
    inp, feat = build_backbone(input_shape)

    # heads
    emo = Dense(num_emotions, activation='softmax', name='emotion')(feat)
    val = Dense(1, activation='sigmoid', name='valence')(feat)
    aro = Dense(1, activation='sigmoid', name='arousal')(feat)

    model = Model(inp, [emo, val, aro])
    model.compile(
        optimizer=Adam(1e-4),
        loss={
            'emotion': focal_loss_multi(alpha_e, gamma=2.0),
            'valence': 'binary_crossentropy',
            'arousal': 'binary_crossentropy',
        },
        loss_weights={'emotion':1.0, 'valence':1.0, 'arousal':1.0},
        metrics={
            'emotion':['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
            'valence':['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')],
            'arousal':['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
        }
    )
    return model


# Experiments Run

In [ ]:
from pathlib import Path

builders = {
    "emotion": lambda input_shape, n_cls, y_tr: create_emotion_model(input_shape, n_cls, y_tr),
    "valence": lambda input_shape, n_cls, y_tr: create_valence_model(input_shape),
    "arousal": lambda input_shape, n_cls, y_tr: create_arousal_model(input_shape),
    "multi":   lambda input_shape, n_cls, y_tr: create_multi_task_cnn(input_shape, n_cls, y_tr),
}

run_experiments(
    root_dir        = Path("/content/drive/MyDrive/ppg_dataset"),
    second_dir      = Path("/content/drive/MyDrive/ppg_dataset_custom"),
    window_sizes    = [500, 1000, 100],
    tasks           = ["emotion","valence","arousal","multi"],
    qc_pctiles      = (5, 5, 5, 95),
    augment_factors = {0: 5, 3: 5, "other": 2},
    augment_funcs   = [add_noise, jitter, time_shift, time_warp, freq_mask],
    builders        = builders,
    output_root     = Path("/content/drive/MyDrive/THESIS/MIXED"),
    overlap         = 0.5,
    train_frac      = 0.8,
    val_frac        = 0.1,
    batch_size      = 32,
    epochs          = 250,
    seed            = 42
)

# Testing

In [ ]:
raw_signals, raw_labels = load_ppg_data('/content/drive/MyDrive/THESIS/ppg_dataset')
signals = [preprocess_ppg(sig) for sig in raw_signals]
labels_enc = LabelEncoder().fit_transform(raw_labels)

_, _, _, _, _, _, _ = prepare_window_data(
              signals,
              labels_enc,
              window_size     = 100,
              overlap         = 0.5,
              train_frac      = 0.8,
              val_frac        = 0.1,
              qc_pctiles      =  (5, 5, 5, 95),
              augment_factors = {},
              augment_funcs   = [],
              seed            = 42
          )

In [ ]:


raw_signals, raw_labels = load_ppg_data('/content/drive/MyDrive/THESIS/ppg_dataset_custom')
signals = [resample_signal(preprocess_ppg(sig, fs=176.0), 176, 100) for sig in raw_signals]
labels_enc = LabelEncoder().fit_transform(raw_labels)

_, _, _, _, _, _, _ = prepare_window_data(
              signals,
              labels_enc,
              window_size     = 500,
              overlap         = 0.5,
              train_frac      = 0.8,
              val_frac        = 0.1,
              qc_pctiles      =  (5, 5, 5, 95),
              augment_factors = {},
              augment_funcs   = [],
              seed            = 42
          )

In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import Model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
import json


base_dir    = Path('/content/drive/MyDrive/THESIS/ppg_dataset_custom')
window_size = 100
overlap     = 0.5
qc_pctiles  = (5,5,5,95)
K_cal_target= 40
task        = 'emotion'  # 'emotion','valence','arousal' or 'multi'
scaler      = joblib.load('scaler.joblib')
model_path  = f'/content/drive/MyDrive/THESIS/{task}_{window_size}/{task}_{window_size}/{task}_{window_size}.keras'

model        = tf.keras.models.load_model(model_path, custom_objects={'loss_fn': focal_loss_multi})
embed_model  = Model(model.input, model.get_layer(task).input if task != 'multi' else model.get_layer('emotion').input)

if task == 'multi':
  head_models  = {
      'emotion': Model(model.input, model.get_layer('emotion').output),
      'valence': Model(model.input, model.get_layer('valence').output),
      'arousal': Model(model.input, model.get_layer('arousal').output)
  }
else:
  head_models  = {task: model}

def chrono_split_windows(signals, labels, frac):
    Xc,yc,Xt,yt = [],[],[],[]
    for sig,lbl in zip(signals,labels):
        cut = int(len(sig)*frac)
        for w in sliding_windows(sig[:cut], window_size, overlap):
            Xc.append(w); yc.append(lbl)
        for w in sliding_windows(sig[cut:], window_size, overlap):
            Xt.append(w); yt.append(lbl)
    return np.array(Xc)[...,None],np.array(yc),np.array(Xt)[...,None],np.array(yt)

def metrics_dict(y_true, y_pred, suffix, n_classes):
    avg = 'weighted'
    d = {
        f'accuracy_{suffix}': accuracy_score(y_true,y_pred),
        f'precision_{suffix}': precision_score(y_true,y_pred,average=avg,zero_division=0),
        f'recall_{suffix}':    recall_score(y_true,y_pred,average=avg,zero_division=0),
        f'f1_{suffix}':        f1_score(y_true,y_pred,average=avg,zero_division=0)
    }
    pr,rc,f1s,_ = precision_recall_fscore_support(y_true,y_pred,labels=list(range(n_classes)),zero_division=0,average=None)
    for c in range(n_classes):
        d[f'class{c}_precision_{suffix}'] = pr[c]
        d[f'class{c}_recall_{suffix}']    = rc[c]
        d[f'class{c}_f1_{suffix}']        = f1s[c]
    return d

def proto_metrics(signals, labels, embed_model, head_model, n_classes, binary):
    step = int(window_size*(1-overlap))
    prelim_frac = min(1.0,(K_cal_target*step)/sum(len(s) for s in signals))
    Xc,yc,Xt,yt = chrono_split_windows(signals,labels,prelim_frac)
    cuts        = compute_cuts(Xc,qc_pctiles)
    mask_c,mask_t = qc_filter(Xc,cuts),qc_filter(Xt,cuts)
    Xc,yc,Xt,yt = Xc[mask_c],yc[mask_c],Xt[mask_t],yt[mask_t]
    Xt_s = scaler.transform(Xt.reshape(-1,window_size)).reshape(-1,window_size,1)
    preds= head_model.predict(Xt_s,verbose=0)
    if binary:
        y0 = (preds.ravel()>0.5).astype(int)
    else:
        y0 = preds.argmax(axis=1)
    res = metrics_dict(yt,y0,'zero',n_classes)
    counts = np.bincount(yc,minlength=n_classes)
    K_eff  = min(K_cal_target, counts.sum())
    quotas = [min(int(round(K_eff*c/counts.sum())),c) for c in counts]
    diff   = K_eff - sum(quotas)
    cap    = sorted(range(n_classes), key=lambda c:counts[c]-quotas[c], reverse=True)
    idx=0; cal_idx=[]
    while diff>0:
        c = cap[idx%n_classes]
        if quotas[c]<counts[c]:
            quotas[c]+=1; diff-=1
        idx+=1
    for c,q in enumerate(quotas):
        idxs = np.where(yc==c)[0]
        if q: cal_idx.extend(idxs[np.linspace(0,len(idxs)-1,q,dtype=int)])
    cal_idx = sorted(cal_idx)
    Xc_s,yc_s = Xc[cal_idx],yc[cal_idx]
    Xc_s = scaler.transform(Xc_s.reshape(-1,window_size)).reshape(-1,window_size,1)
    Zc = embed_model.predict(Xc_s,verbose=0)
    Zc/= np.linalg.norm(Zc,axis=1,keepdims=True)
    prot = np.vstack([Zc[yc_s==c].mean(axis=0) for c in range(n_classes)])
    prot/= np.linalg.norm(prot,axis=1,keepdims=True)
    Zt = embed_model.predict(Xt_s,verbose=0)
    Zt/= np.linalg.norm(Zt,axis=1,keepdims=True)
    y1 = (Zt @ prot.T).argmax(axis=1)
    res.update(metrics_dict(yt,y1,'cal',n_classes))
    return res

records=[]
for subj in sorted(base_dir.iterdir()):
    if not subj.is_dir(): continue
    raw_sigs,raw_lbls = load_ppg_data(str(subj),recursive=True)
    sigs = [resample_signal(preprocess_ppg(s),176,100) for s in raw_sigs]
    tasks = ['emotion','valence','arousal'] if task=='multi' else [task]
    for t in tasks:
        lbls = LabelEncoder().fit_transform(raw_lbls)
        if t=='valence':
            m={0:0,1:1,2:1,3:0}; lbls=np.array([m[int(c)] for c in lbls])
        if t=='arousal':
            m={0:1,1:1,2:0,3:0}; lbls=np.array([m[int(c)] for c in lbls])
        n_cls  = 4 if t=='emotion' else 2
        binary = (t!='emotion')
        rec = proto_metrics(sigs,lbls,embed_model,head_models[t],n_cls,binary)
        rec.update(subject=subj.name, task=t)
        records.append(rec)

df = pd.DataFrame(records)

for t, group in df.groupby('task'):
    n_cls = 4 if t=='emotion' else 2
    metrics = ['accuracy','precision','recall','f1'] \
            + [f'class{c}_{m}' for c in range(n_cls) for m in ('precision','recall','f1')]
    cols = ['subject','task'] \
         + [f'{m}_{sfx}' for m in metrics for sfx in ('zero','cal')]
    sub = group[cols]  # now only asks for class0/1 for valence/arousal
    ms  = sub.drop(columns=['subject','task']).agg(['mean','std']).T
    ms['mean±std'] = ms['mean'].round(4).astype(str) + ' ± ' + ms['std'].round(4).astype(str)
    print(f"\nTask: {t}")
    print(ms['mean±std'].to_string())


df.to_csv(f"/content/drive/MyDrive/THESIS/{task}_{window_size}/custom_dataset_metrics_{task}_{window_size}.csv")
with open(f'/content/drive/MyDrive/THESIS/{task}_{window_size}/custom_dataset_stats_{task}_{window_size}.txt','w') as f:
    f.write(ms['mean±std'].to_string())

df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RAW_SR, TARGET_SR = 176, 100
DUR_SEC = 10
N_RAW = RAW_SR * DUR_SEC

# Assume signals dict exists
raw9 = signals['subject_9']

# Extract first 10 s raw segment
raw_seg = raw9[:N_RAW]

# Process & resample that exact segment
proc_seg = resample_signal(preprocess_ppg(pd.Series(raw_seg)), RAW_SR, TARGET_SR)

# Create time axes matched to each segment’s actual length
t_raw = np.arange(len(raw_seg))  / RAW_SR
t_proc = np.arange(len(proc_seg)) / TARGET_SR

# Plot & save raw
plt.figure(figsize=(10,3))
plt.plot(t_raw, raw_seg)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Subject 9 — Raw PPG (first 10 s)")
plt.tight_layout()
plt.savefig('subject9_raw_10s.png')
plt.close()

# Plot & save processed
plt.figure(figsize=(10,3))
plt.plot(t_proc, proc_seg)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Subject 9 — Processed PPG (first 10 s)")
plt.tight_layout()
plt.savefig('subject9_processed_10s.png')
plt.close()


# Dataset Analysis

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/ppg_dataset_custom'

# 2) Imports
import os, glob, pandas as pd

RAW_SR = 176  # original sampling rate

# 3) Load all sessions into a DataFrame
def load_sessions(root):
    rows = []
    for subj in sorted(os.listdir(root)):
        subj_path = os.path.join(root, subj)
        if os.path.isdir(subj_path):
            for f in glob.glob(os.path.join(subj_path, '*', '*.txt')):
                name  = os.path.splitext(os.path.basename(f))[0]
                label = name.split('_')[0]
                sig   = pd.read_csv(f, sep='\t', header=None, names=['signal'])['signal']
                rows.append({
                    'subject': subj,
                    'session': name,
                    'label':   label,
                    'raw_len': len(sig)
                })
    return pd.DataFrame(rows)

df = load_sessions(DATA_ROOT)

# 4) Compute raw durations
df['duration_s']   = df['raw_len'] / RAW_SR
df['duration_min'] = df['duration_s'] / 60

# 5) Per‐subject summary
subj_meta = df.groupby('subject').duration_min.agg(
    mean_min = 'mean',
    std_min  = 'std',
    total_min= 'sum',
    sessions = 'count'
).reset_index()

# 6) Per‐emotion summary
emo_meta = df.groupby('label').duration_min.agg(
    mean_min   = 'mean',
    std_min    = 'std',
    total_min  = 'sum',
    recordings = 'count'
).reset_index()

# 7) Optionally convert to mm:ss strings
def to_mmss(x):
    total_s = int(round(x*60))
    m, s = divmod(total_s, 60)
    return f"{m:d}:{s:02d}"

subj_meta['mean_mmss']  = subj_meta['mean_min'].apply(to_mmss)
subj_meta['std_mmss']   = subj_meta['std_min'].apply(to_mmss)
subj_meta['total_mmss'] = subj_meta['total_min'].apply(to_mmss)

emo_meta['mean_mmss']   = emo_meta['mean_min'].apply(to_mmss)
emo_meta['std_mmss']    = emo_meta['std_min'].apply(to_mmss)
emo_meta['total_mmss']  = emo_meta['total_min'].apply(to_mmss)

# 8) Save results
subj_meta.to_csv('subject_duration_metadata_raw.csv', index=False)
emo_meta.to_csv('emotion_duration_summary_raw.csv', index=False)
df.to_csv('session_durations_raw.csv', index=False)

# 9) Preview
print("Per‐subject durations:")
print(subj_meta[['subject','mean_mmss','std_mmss','total_mmss','sessions']])
print("\nPer‐emotion durations:")
print(emo_meta[['label','mean_mmss','std_mmss','total_mmss','recordings']])

In [ ]:
from google.colab import drive
DATA_ROOT = '/content/drive/MyDrive/ppg_dataset_custom'

import os, glob
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, welch

RAW_SR, TARGET_SR = 176, 100

def load_subjects(root):
    out = {}
    for subj in os.listdir(root):
        p = os.path.join(root, subj)
        if os.path.isdir(p):
            sess = {}
            for f in glob.glob(os.path.join(p, '*', '*.txt')):
                name = os.path.splitext(os.path.basename(f))[0]
                lbl  = name.split('_')[0]
                sig  = pd.read_csv(f, sep='\t', header=None, names=['signal'])['signal']
                sess[name] = {'raw': sig, 'label': lbl}
            out[subj] = sess
    return out

def extract_features(sig):
    p = preprocess_ppg(sig)
    r = resample_signal(p, RAW_SR, TARGET_SR)
    peaks, _ = find_peaks(r, distance=TARGET_SR*0.5)  # min 0.5s between peaks
    ibi = np.diff(peaks) / TARGET_SR * 1000           # IBI in ms
    hr = 60_000 / ibi                                  # instantaneous HR in bpm
    f, psd = welch(ibi, fs=4.0, nperseg=len(ibi)//2)    # PSD of IBI, 4 Hz
    return {
        'mean_hr':            np.nanmean(hr),
        'sdnn':               np.nanstd(ibi),
        'rmssd':              np.sqrt(np.nanmean(np.diff(ibi)**2)),
        'pnn50':              np.mean(np.abs(np.diff(ibi))>50),
        'peak_rate':          len(peaks)/(len(r)/TARGET_SR),
        'ibi_lf_power':       psd[(f>=0.04)&(f<0.15)].sum(),
        'ibi_hf_power':       psd[(f>=0.15)&(f<0.4)].sum(),
        'lf_hf_ratio':        psd[(f>=0.04)&(f<0.15)].sum()/psd[(f>=0.15)&(f<0.4)].sum()
    }

subjects = load_subjects(DATA_ROOT)

rows = []
for subj, sess in subjects.items():
    for name, data in sess.items():
        feats = extract_features(data['raw'])
        feats.update(subject=subj, session=name, label=data['label'])
        rows.append(feats)

df_features = pd.DataFrame(rows)
df_features.to_csv('ppg_time_domain_features.csv', index=False)


In [ ]:
from google.colab import drive
DATA_ROOT = '/content/drive/MyDrive/ppg_dataset_custom'

import os, glob, pandas as pd

RAW_SR, TARGET_SR = 176, 100

def load_sessions(root):
    rows = []
    for subj in os.listdir(root):
        path = os.path.join(root, subj)
        if os.path.isdir(path):
            for f in glob.glob(os.path.join(path, '*', '*.txt')):
                name = os.path.splitext(os.path.basename(f))[0]
                sig  = pd.read_csv(f, sep='\t', header=None, names=['signal'])['signal']
                rows.append({'subject': subj, 'session': name, 'raw': sig})
    return pd.DataFrame(rows)

def compute_durations(sig):
    raw_time  = len(sig) / RAW_SR
    proc      = resample_signal(preprocess_ppg(sig), RAW_SR, TARGET_SR)
    proc_time = len(proc) / TARGET_SR
    return pd.Series({'raw_s': raw_time, 'proc_s': proc_time, 'delta_s': proc_time - raw_time})

df      = load_sessions(DATA_ROOT)
dur_df  = df['raw'].apply(compute_durations)
result  = pd.concat([df[['subject','session']], dur_df], axis=1)

print(result.head(10))

In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = '/content/drive/MyDrive/ppg_dataset_custom'  # adjust if needed
RAW_SR, TARGET_SR = 176, 100
START_SEC, DUR_SEC = 30, 10

def load_signal(fpath):
    # single‐column of amplitude values
    return pd.read_csv(fpath, header=None, names=['signal'])['signal']

# gather one example per emotion
examples = {}
for subj in sorted(os.listdir(DATA_ROOT)):
    subj_p = os.path.join(DATA_ROOT, subj)
    if not os.path.isdir(subj_p): continue
    for sess in os.listdir(subj_p):
        sess_p = os.path.join(subj_p, sess)
        if not os.path.isdir(sess_p): continue
        for f in glob.glob(os.path.join(sess_p, '*.txt')):
            label = os.path.basename(f).split('_')[0]
            if label in examples: continue
            examples[label] = load_signal(f)
            if len(examples) == 4: break
        if len(examples) == 4: break
    if len(examples) == 4: break

# plot raw vs processed for 30–40s window
for label, raw in examples.items():
    # raw segment
    i0, i1 = int(START_SEC*RAW_SR), int((START_SEC+DUR_SEC)*RAW_SR)
    raw_seg = raw.iloc[i0:i1].reset_index(drop=True)
    raw_centered = raw_seg - raw_seg.mean()
    t_raw = raw_centered.index / RAW_SR

    # processed segment
    proc_full = resample_signal(preprocess_ppg(raw), RAW_SR, TARGET_SR)
    j0 = int(START_SEC*TARGET_SR)
    proc_seg = proc_full[j0:j0 + DUR_SEC*TARGET_SR]
    t_proc = proc_seg.index / TARGET_SR

    plt.figure(figsize=(6,2))
    plt.plot(t_raw,  raw_centered, label='Raw (mean‐centered)')
    plt.plot(t_proc, proc_seg,    label='Processed')
    plt.xlim(0, DUR_SEC)
    plt.title(f"PPG: {label.capitalize()} (30–40 s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (a.u.)")
    plt.legend(loc='upper right', frameon=False)
    plt.tight_layout()
    plt.show()


# ON Device Models

In [ ]:
!pip install tensorflow_model_optimization

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical


# CONFIG
BASE_1S       = Path("/content/drive/MyDrive/THESIS/emotion_100/emotion_100/emotion_100.keras")
BASE_5S       = Path("/content/drive/MyDrive/THESIS/emotion_500/emotion_500/emotion_500.keras")
ROOT_WRIST    = Path("/content/drive/MyDrive/THESIS/ppg_dataset_custom")
OUT_ROOT      = Path("/content/drive/MyDrive/THESIS/EMO_ONDEVICE")
WINDOW_SIZES  = [100, 500]    # samples at 100 Hz → 1 s & 5 s
OVERLAP       = 0.5
QC            = (5,5,5,95)
TRAIN_FRAC    = 0.8
VAL_FRAC      = 0.1
BATCH         = 32
EPOCHS_FINE   = 100
SEED          = 42
REP_CALIB     = 200

# callbacks tuned for long fine-tune
fine_tune_cbs = [
    tf.keras.callbacks.ReduceLROnPlateau("val_accuracy", factor=0.5, patience=15, verbose=1),
    tf.keras.callbacks.EarlyStopping("val_accuracy", patience=30, restore_best_weights=True, verbose=1)
]

# 1) Load & preprocess wrist data
raw_sigs, raw_lbls = load_ppg_data(str(ROOT_WRIST), recursive=True)
proc_wrist = [
    resample_signal(preprocess_ppg(s, fs=176.0), 176, 100)
    for s in raw_sigs
]
labels_int = LabelEncoder().fit_transform(raw_lbls)

for win in WINDOW_SIZES:
    out_dir = OUT_ROOT / f"emotion_{win}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # 2) Load base fingertip model (no compile)
    base_path = BASE_1S if win == 100 else BASE_5S
    model = tf.keras.models.load_model(str(base_path), compile=False)

    # 3) Window & split wrist data
    X_tr, y_tr, X_val, y_val, _, _, n_cls = prepare_window_data(
        signals         = proc_wrist,
        labels          = labels_int,
        window_size     = win,
        overlap         = OVERLAP,
        train_frac      = TRAIN_FRAC,
        val_frac        = VAL_FRAC,
        qc_pctiles      = QC,
        augment_factors = {},
        augment_funcs   = [],
        seed            = SEED
    )

    # 4) Compile for fine-tuning with focal loss
    freqs = np.bincount(y_tr, minlength=n_cls)
    alpha = freqs.sum() / (n_cls * freqs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=focal_loss_multi(alpha, gamma=2.0),
        metrics=["accuracy"]
    )

    # 5) One-hot encode & fine-tune
    y_tr_ohe  = to_categorical(y_tr,  num_classes=n_cls)
    y_val_ohe = to_categorical(y_val, num_classes=n_cls)

    hist = model.fit(
        tf.data.Dataset.from_tensor_slices((X_tr, y_tr_ohe))
          .shuffle(len(X_tr)).batch(BATCH),
        validation_data = tf.data.Dataset.from_tensor_slices((X_val, y_val_ohe)).batch(BATCH),
        epochs     = EPOCHS_FINE,
        callbacks  = fine_tune_cbs,
        verbose    = 1
    )
    pd.DataFrame(hist.history).to_csv(out_dir / f"history_finetune_{win}.csv", index=False)
    model.save(out_dir / f"emotion_{win}_finetuned.keras")

    # 6) Evaluate on wrist validation set
    preds = model.predict(X_val).argmax(axis=1)
    pd.DataFrame(classification_report(y_val, preds, output_dict=True)).T \
      .to_csv(out_dir / f"class_report_finetune_{win}.csv")
    np.savetxt(
        out_dir / f"confusion_matrix_finetune_{win}.csv",
        confusion_matrix(y_val, preds),
        fmt="%d"
    )

    # 7) Convert directly to full-integer TFLite
    def rep_ds():
        for i in np.random.choice(len(X_tr), REP_CALIB, replace=False):
            yield [tf.cast(X_tr[i:i+1], tf.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type      = tf.int8
    converter.inference_output_type     = tf.int8
    converter.representative_dataset    = rep_ds
    tfl = converter.convert()
    (out_dir / f"emotion_{win}_finetuned_int8.tflite").write_bytes(tfl)


In [ ]:
import numpy as np
from pathlib import Path
from joblib import dump
from sklearn.preprocessing import StandardScaler, LabelEncoder


ROOT_FINGER    = Path("/content/drive/MyDrive/THESIS/ppg_dataset")
ROOT_WRIST     = Path("/content/drive/MyDrive/THESIS/ppg_dataset_custom")
OVERLAP        = 0.5
QC             = (5,5,5,95)
TRAIN_FRAC     = 0.8
VAL_FRAC       = 0.1
SEED           = 42

s_finger, l_finger = load_ppg_data(str(ROOT_FINGER))
s_wrist,  l_wrist  = load_ppg_data(str(ROOT_WRIST))

proc_finger = [preprocess_ppg(s, fs=100.0) for s in s_finger]
proc_wrist  = [resample_signal(preprocess_ppg(s, fs=176.0), 176, 100) for s in s_wrist]

labels_finger_int = LabelEncoder().fit_transform(l_finger)
labels_wrist_int  = LabelEncoder().fit_transform(l_wrist)

for prefix, signals, labels in [
    ("finger", proc_finger, labels_finger_int),
    ("wrist",  proc_wrist,  labels_wrist_int)
]:
    for win in (100, 500):
        X_tr, *_ = prepare_window_data(
            signals         = signals,
            labels          = labels,
            window_size     = win,
            overlap         = OVERLAP,
            train_frac      = TRAIN_FRAC,
            val_frac        = VAL_FRAC,
            qc_pctiles      = QC,
            augment_factors = {},
            augment_funcs   = [],
            seed            = SEED
        )
        scaler = StandardScaler().fit(X_tr.reshape(-1, win))
        dump(scaler, f"/content/drive/MyDrive/THESIS/scaler_{prefix}_{win}.joblib")


In [ ]:
from joblib import load
import numpy as np

# --- IMPORTANT ---
# Replace 'path/to/your/scaler.joblib' with the actual path to your file
SCALER_FILE_PATH = 'scaler (2).joblib'

try:
    # Load the scaler object from the .joblib file
    scaler = load(SCALER_FILE_PATH)

    # Access the mean and scale (standard deviation) attributes
    # These attributes are available after the scaler has been fitted.

    if hasattr(scaler, 'mean_') and hasattr(scaler, 'scale_'):
        scaler_mean = scaler.mean_
        scaler_std = scaler.scale_ # scale_ is the standard deviation

        print(f"Scaler loaded successfully from: {SCALER_FILE_PATH}")
        print("-" * 30)

        print("Scaler Mean (scaler.mean_):")
        # print(scaler_mean)
        # For direct copy-pasting into JavaScript, format as a JS array string
        print("JavaScript array format for mean_:")
        print(f"[{', '.join(map(str, scaler_mean))}]")
        print("-" * 30)

        print("Scaler Standard Deviation (scaler.scale_):")
        # print(scaler_std)
        # For direct copy-pasting into JavaScript, format as a JS array string
        print("JavaScript array format for scale_:")
        print(f"[{', '.join(map(str, scaler_std))}]")
        print("-" * 30)

        print(f"Number of features in mean: {len(scaler_mean)}")
        print(f"Number of features in std: {len(scaler_std)}")

        # You can now copy these printed arrays and paste them into the
        # 'setScalerValues()' method in your EmotionRecognitionService.js

    else:
        print(f"Error: The loaded object from {SCALER_FILE_PATH} does not appear to be a fitted StandardScaler.")
        print("Make sure the scaler was fitted before saving, and the path is correct.")
        if hasattr(scaler, 'get_params'):
             print("Loaded object parameters:", scaler.get_params())


except FileNotFoundError:
    print(f"Error: Scaler file not found at '{SCALER_FILE_PATH}'. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading or inspecting the scaler: {e}")